In [139]:
import json

dataset = []
with open("../data/hsp_step_accuracies.jsonl", 'r') as f:
    for line in f:
        dataset.append(json.loads(line))

In [140]:
# check the rankings of sentences by integrated gradients
with open("../data/attributions/result3.1.json", 'r') as f:
    ig_attributions = json.load(f)

In [159]:
import time
import numpy as np
from typing import Dict, List, Tuple, Optional

def find_subsequence(
    tokens,
    subtokens,
    start_from,
):
    sublen = len(subtokens)
    # Iterate through each token pair
    for i in range(start_from, len(tokens) - sublen+1):
        if tokens[i:i+sublen] == subtokens:
            return (i, i+sublen)
    return None

def align_steps_to_tokens(
    tokens,
    attributions,
    steps,
    tokenizer,
    start_from
):
    step_data = []
    search_start = start_from
    
    for step_idx, step_text in enumerate(steps):
        step_text += "\n"
        # Tokenize this step
        step_token_ids = tokenizer.encode(step_text, add_special_tokens=False)
        step_tokens = [tokenizer.convert_ids_to_tokens(tid) for tid in step_token_ids]
        
        # Find this subsequence in the full token list
        match = find_subsequence(tokens, step_tokens, start_from=search_start)
        
        if match is None:
            print(f"Warning: Could not find step {step_idx} in token sequence")
            print(f"  Step text: {step_text[:100]}...")
            print(f"  Step tokens: {step_tokens[:10]}...")
            step_data.append({
                'step_idx': step_idx,
                'step_text': step_text,
                'matched': False
            })
            continue
        
        start_idx, end_idx = match
        step_attrs = attributions[start_idx:end_idx]
        
        # Compute metrics following the paper's approach
        abs_attrs = np.abs(step_attrs)
        strength = float(np.sum(abs_attrs))
        normalized_strength = strength / np.sqrt(len(step_attrs)) if len(step_attrs) > 0 else 0.0
        
        signed_sum = float(np.sum(step_attrs))
        consistency = abs(signed_sum) / strength if strength > 0 else 0.0
        
        step_data.append({
            'step_idx': step_idx,
            'step_text': step_text,
            'matched': True,
            'token_start': start_idx,
            'token_end': end_idx,
            'n_tokens': end_idx - start_idx,
            'tokens': tokens[start_idx:end_idx],
            'attributions': step_attrs,
            'strength': strength,
            'normalized_strength': normalized_strength,
            'consistency': consistency,
            'mean_attribution': float(np.mean(step_attrs)) if len(step_attrs) > 0 else 0.0,
            'signed_sum': signed_sum
        })
        
        # Move search start forward to avoid re-matching
        search_start = end_idx
    
    return step_data

def normalize_step_strengths(step_data: List[Dict]) -> List[Dict]:
    """Normalize strengths across all steps (like the paper's Eq. 4)."""
    total_strength = sum(s['normalized_strength'] for s in step_data if s.get('matched', False))
    
    for step in step_data:
        if step.get('matched', False) and total_strength > 0:
            step['normalized_strength_global'] = step['normalized_strength'] / total_strength
        else:
            step['normalized_strength_global'] = 0.0
    
    return step_data

In [160]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")  # or whatever you used

# Your ground truth steps (list of strings)
ground_truth_steps = dataset[2]['steps'][:dataset[2]['first_hsp_index']+1]

# Your IG results
tokens = ig_attributions['tokens']
attributions = ig_attributions['attributions']

step_data = align_steps_to_tokens(tokens, attributions, ground_truth_steps, tokenizer, start_from=92)
step_data = normalize_step_strengths(step_data)

In [161]:
ig_attribution_data = {
    "problem_id": 2,
    "method": "integrated_gradients",
    "sentence_scores" : {},
}

for i, step in enumerate(step_data):
    ig_attribution_data['sentence_scores'][i] = step['normalized_strength_global']

In [162]:
with open("../data/attributions/problem3_attribution_scores.json", 'w') as f:
    json.dump(ig_attribution_data, f)

In [163]:
ranked_by_ig = sorted(
    [s for s in step_data if s['matched']],
    key=lambda x: x['normalized_strength_global'],
    reverse=True
)

print("Steps ranked by IG attribution (highest first):")
print("-"*60)
for rank, step in enumerate(ranked_by_ig):
    print(f"Rank {rank+1}: Step: {step['step_idx']} | "
          f"strength={step['normalized_strength_global']:.4f} | "
          f"consistency={step['consistency']:.3f} | "
          f"text={step['step_text']}")


Steps ranked by IG attribution (highest first):
------------------------------------------------------------
Rank 1: Step: 4 | strength=0.1221 | consistency=0.387 | text=Then for each k ≥ 1, the absolute value of x_k is equal to the absolute value of x_{k-1} + 3.

Rank 2: Step: 55 | strength=0.0917 | consistency=0.979 | text=Expanding the right-hand side: x_{k-1}² + 6x_{k-1} + 9.

Rank 3: Step: 44 | strength=0.0821 | consistency=0.964 | text=Let me try:

S_n = S_{n-1} + x_n.

Rank 4: Step: 5 | strength=0.0718 | consistency=0.984 | text=So, for each term, we have two possibilities: x_k = x_{k-1} + 3 or x_k = -(x_{k-1} + 3).

Rank 5: Step: 56 | strength=0.0597 | consistency=0.914 | text=Therefore, x_k² - x_{k-1}² = 6x_{k-1} + 9.

Rank 6: Step: 1 | strength=0.0418 | consistency=0.161 | text=+ x₂₀₀₆| given that x₀ = 0 and |x_k| = |x_{k-1} + 3| for all integers k ≥ 1.

Rank 7: Step: 0 | strength=0.0307 | consistency=0.884 | text=Okay, so I need to find the minimum possible value of |x₁ + x₂

In [164]:
# check the rankings of sentences by attn
with open("../data/attention_attribution_results/problem_2_attention.json", 'r') as f:
    attn_attributions = json.load(f)

In [165]:
# Explore the attention attribution structure
print("Keys:", attn_attributions.keys())
print("\nProblem ID:", attn_attributions['problem_id'])
print("ESSP index:", attn_attributions['essp_index'])
print("Num sentences:", attn_attributions['num_sentences'])

# Check the structure of the scores
print("\nRaw scores type:", type(attn_attributions['raw_scores']))
print("Raw scores length:", len(attn_attributions['raw_scores']) if isinstance(attn_attributions['raw_scores'], list) else "N/A")
print("Raw scores sample:", attn_attributions['raw_scores'][:5] if isinstance(attn_attributions['raw_scores'], list) else attn_attributions['raw_scores'])

print("\nNormalized scores type:", type(attn_attributions['normalized_scores']))
print("Normalized scores length:", len(attn_attributions['normalized_scores']) if isinstance(attn_attributions['normalized_scores'], list) else "N/A")
print("Normalized scores sample:", attn_attributions['normalized_scores'][:5] if isinstance(attn_attributions['normalized_scores'], list) else attn_attributions['normalized_scores'])

# Check sentence_attributions structure
print("\nSentence attributions type:", type(attn_attributions['sentence_attributions']))
if isinstance(attn_attributions['sentence_attributions'], list):
    print("Length:", len(attn_attributions['sentence_attributions']))
    print("First item:", attn_attributions['sentence_attributions'][0] if len(attn_attributions['sentence_attributions']) > 0 else "empty")


Keys: dict_keys(['problem_id', 'problem_index', 'essp_index', 'num_sentences', 'sentence_attributions', 'raw_scores', 'normalized_scores'])

Problem ID: 2
ESSP index: 258
Num sentences: 259

Raw scores type: <class 'list'>
Raw scores length: 259
Raw scores sample: [3.0596034959192797e-15, 1.934254596304807e-24, 1.0051648520461534e-32, 1.3740810498965318e-30, 1.658148940742474e-24]

Normalized scores type: <class 'list'>
Normalized scores length: 259
Normalized scores sample: [7.864346984771135e-14, 4.9717714476787023e-23, 2.5836567332761486e-31, 3.531911953950141e-29, 4.262074690338752e-23]

Sentence attributions type: <class 'list'>
Length: 259
First item: {'tokens': ['Okay', ',', 'Ġso', 'ĠI', 'Ġneed', 'Ġto', 'Ġfind', 'Ġthe', 'Ġminimum', 'Ġpossible', 'Ġvalue', 'Ġof', 'Ġ|', 'x', 'âĤģ', 'Ġ+', 'Ġx', 'âĤĤ', 'Ġ+', 'Ġ...'], 'attention_raw': 3.0596034959192797e-15, 'attention_normalized': 7.864346984771135e-14, 'token_range': [92, 112], 'orig_idx': 0}


In [166]:
first_hsp_index = dataset[2]['first_hsp_index']
attn_scores = attn_attributions['normalized_scores'][:first_hsp_index + 1]

attn_step_data = []
for step_idx, score in enumerate(attn_scores):
    attn_step_data.append({
        'step_idx': step_idx,
        'attention_score': score,
        'step_text': dataset[2]['steps'][step_idx]
    })
    
ranked_by_attn = sorted(
    attn_step_data,
    key=lambda x: x['attention_score'],
    reverse=True
)

print("Steps ranked by attention attribution (highest first):")
print("-"*60)
for rank, step in enumerate(ranked_by_attn):
    print(f"Rank {rank+1}: Step {step['step_idx']} | "
          f"attn_score={step['attention_score']} | "
          f"text={step['step_text']}")

Steps ranked by attention attribution (highest first):
------------------------------------------------------------
Rank 1: Step 0 | attn_score=7.864346984771135e-14 | text=Okay, so I need to find the minimum possible value of |x₁ + x₂ + ...
Rank 2: Step 34 | attn_score=4.755141007111972e-18 | text=We need to find S_2006 with minimal |S_2006|.
Rank 3: Step 1 | attn_score=4.9717714476787023e-23 | text=+ x₂₀₀₆| given that x₀ = 0 and |x_k| = |x_{k-1} + 3| for all integers k ≥ 1.
Rank 4: Step 4 | attn_score=4.262074690338752e-23 | text=Then for each k ≥ 1, the absolute value of x_k is equal to the absolute value of x_{k-1} + 3.
Rank 5: Step 82 | attn_score=2.9491444082544203e-23 | text=Alternatively, since S_n = S_{n-1} + x_n, and x_n = ±(x_{n-1} + 3), then:

S_n = S_{n-1} ± (x_{n-1} + 3).
Rank 6: Step 9 | attn_score=9.575510966005848e-24 | text=So, the challenge is to figure out a sequence of signs (either + or -) for each term such that when you add up all these terms, the total is as cl

In [167]:
attn_attribution_data = {
    "problem_id": 2,
    "method": "attention",
    "sentence_scores" : {},
}

for i, step in enumerate(attn_step_data):
    attn_attribution_data['sentence_scores'][i] = step['attention_score']

In [168]:
with open("../data/attention_attribution_results/problem2_attention_scores.json", 'w') as f:
    json.dump(attn_attribution_data, f)